In [7]:
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

In [79]:
class ConvReg(nn.Module):
    def __init__(self):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=8,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 1000 -> 500

            nn.Conv1d(
                in_channels=8,
                out_channels=16,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 500 -> 250

            nn.Conv1d(
                in_channels=16,
                out_channels=32,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 250 -> 125

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 125 -> 62
            
            nn.Conv1d(
                in_channels=64,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 62 -> 31
            
            nn.Conv1d(
                in_channels=64,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2),   # 31 -> 15
            
            # 전체 시간축 평균 pooling
            nn.AdaptiveAvgPool1d(1)        # (B, 64, 15) -> (B, 64, 1)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),                  # (B, 64, 1) -> (B, 64)
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2)               # output = [SBP, DBP]
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.regressor(x)
        return x

In [80]:
model = ConvReg()

summary(
    model,
    input_size=(1, 1, 1000)  # batch_size=1, channel=1, length=1000
)

Layer (type:depth-idx)                   Output Shape              Param #
ConvReg                                  [1, 2]                    --
├─Sequential: 1-1                        [1, 64, 1]                --
│    └─Conv1d: 2-1                       [1, 8, 1000]              48
│    └─BatchNorm1d: 2-2                  [1, 8, 1000]              16
│    └─ReLU: 2-3                         [1, 8, 1000]              --
│    └─AvgPool1d: 2-4                    [1, 8, 500]               --
│    └─Conv1d: 2-5                       [1, 16, 500]              656
│    └─BatchNorm1d: 2-6                  [1, 16, 500]              32
│    └─ReLU: 2-7                         [1, 16, 500]              --
│    └─AvgPool1d: 2-8                    [1, 16, 250]              --
│    └─Conv1d: 2-9                       [1, 32, 250]              2,592
│    └─BatchNorm1d: 2-10                 [1, 32, 250]              64
│    └─ReLU: 2-11                        [1, 32, 250]              --
│    └─AvgP